In [ ]:
# 1. Hugging Face login FIRST, so the whole run is hands-free after this
# one paste. Read access is enough here; write access only matters for the
# commented publish block at the end.
from huggingface_hub import login

login()

# Text-to-SQL: Qwen3.5-4B phase-1 SFT

Fine-tune `Qwen/Qwen3.5-4B` (the Apprentice house base, Apache-2.0) with LoRA on 20,000 SynSQL-2.5M rows, then measure BIRD-dev execution accuracy (EX). Every run-derived number is printed. Continue gate: **66.0 EX**. Colab T4 uses 4-bit loading; A100 uses bf16. Run top to bottom.

Full BIRD dev is 1,534 questions: two full generation passes (baseline + fine-tuned) plus training take several hours on T4. Use A100 (Colab Pro) for the publishable full run; set `QUICK_N = 200` in cell 5 for a smoke pass first.

## Install pinned training dependencies

Pins match the CUAD notebook. These are the last versions before an XET downloader change that fails on Colab. Colab-provided PyTorch stays in place.

In [ ]:
# 2. Install (Colab)
!pip install -q unsloth==2026.6.9 unsloth-zoo==2026.6.7 ijson

## Fetch harness

Clone public repository so data preparation and BIRD scoring use committed code.

In [ ]:
# 3. Clone once per fresh runtime.
from pathlib import Path
import subprocess

REPO = Path("/content/apprentice-benchmark")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/singhabhishekkk/apprentice-benchmark.git", str(REPO)], check=True)
print(f"repo: {REPO}")

## Prepare SynSQL training subset

`--rows 20000` is sized for Colab and is tunable. SynSQL ships as one large `data.json` (~10 GB download, cached) plus `tables.json` schemas; the script streams both with `ijson` and reservoir-samples with seed 42 (the file is grouped by database, so head-sampling would be biased). Expect the download plus a few minutes of scanning on the first run.

In [ ]:
# 4. Build training data.
TRAIN_JSONL = Path("/content/train.jsonl")
subprocess.run([
    "python", str(REPO / "tasks/text-to-sql/prepare_data.py"),
    "--rows", "20000", "--output", str(TRAIN_JSONL),
], check=True)

## Place BIRD dev data manually

Download BIRD dev zip from [bird-bench.github.io](https://bird-bench.github.io/), then unzip under `/content/bird`. Expected layout is either `/content/bird/dev.json` plus `/content/bird/dev_databases/`, or the same pair inside `/content/bird/dev_20240627/`. No stable direct download URL is assumed.

Optional Drive workflow (leave commented unless needed):
```python
# from google.colab import drive
# drive.mount('/content/drive')
# BIRD_DIR = Path('/content/drive/MyDrive/bird')
```

In [ ]:
# 5. Validate BIRD layout and choose full or smoke-only evaluation.
import importlib.util
import json
import os
import shutil
import sys

BIRD_DIR = Path("/content/bird")
QUICK_N = None  # None = full 1,534-row BIRD dev; use 200 only for smoke runs.
EVAL_SCRIPT = REPO / "tasks/text-to-sql/eval_bird.py"
spec = importlib.util.spec_from_file_location("eval_bird", EVAL_SCRIPT)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Cannot import {EVAL_SCRIPT}")
eval_bird = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = eval_bird
spec.loader.exec_module(eval_bird)

try:
    dev_json, databases = eval_bird.find_layout(BIRD_DIR)
    all_tasks = eval_bird.load_dev(dev_json, databases)
except SystemExit as error:
    raise RuntimeError(
        "BIRD dev missing. Download dev zip from https://bird-bench.github.io/ "
        "and unzip it under /content/bird as documented above."
    ) from error

if QUICK_N is not None:
    if not 1 <= QUICK_N <= len(all_tasks):
        raise ValueError(f"QUICK_N must be 1..{len(all_tasks)} or None")
    print("WARNING: QUICK_N ACTIVE — SMOKE-ONLY RESULT; DO NOT PUBLISH.")
    eval_root = Path("/content/bird_quick")
    eval_root.mkdir(exist_ok=True)
    (eval_root / "dev.json").write_text(json.dumps(all_tasks[:QUICK_N]), encoding="utf-8")
    quick_databases = eval_root / "dev_databases"
    if quick_databases.is_symlink() or quick_databases.exists():
        if quick_databases.is_dir() and not quick_databases.is_symlink():
            shutil.rmtree(quick_databases)
        else:
            quick_databases.unlink()
    os.symlink(databases, quick_databases, target_is_directory=True)
    EVAL_BIRD_DIR = eval_root
else:
    EVAL_BIRD_DIR = BIRD_DIR

eval_dev_json, eval_databases = eval_bird.find_layout(EVAL_BIRD_DIR)
eval_tasks = eval_bird.load_dev(eval_dev_json, eval_databases)
print(f"BIRD eval rows: {len(eval_tasks)}; layout: {eval_dev_json.parent}")

## Load raw base model

T4 loads 4-bit; A100 loads bf16. LoRA is attached only after baseline evaluation, preserving an honest raw-model floor.

In [ ]:
# 6. Load Qwen3.5-4B.
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "Qwen/Qwen3.5-4B"
# Some BIRD schemas exceed 4k tokens as DDL; an 8k cap avoids a mid-eval
# crash on long-schema questions and still fits batch-1 generation on T4.
MAX_SEQ_LENGTH = 8192
gpu_name = torch.cuda.get_device_name(0)
use_bf16 = torch.cuda.is_bf16_supported() and "A100" in gpu_name
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=not use_bf16,
    dtype=torch.bfloat16 if use_bf16 else None,
)
print(f"GPU: {gpu_name}; precision: {'bf16' if use_bf16 else '4-bit'}")

## Baseline: raw model on BIRD dev

Prompts contain sqlite schema DDL, BIRD evidence, question, and `Output ONLY the SQL`. Harness helpers remain the source of truth for layout, schema extraction, and fence stripping. Predictions are flushed row-by-row.

In [ ]:
# 7. Shared prompt and generation path for baseline and final evaluation.
def sql_prompt(schema, question, evidence=""):
    return (
        f"Schema:\n{schema}\n\n"
        f"External knowledge/evidence:\n{evidence}\n\n"
        f"Question:\n{question}\n\nOutput ONLY the SQL."
    )

def generate_predictions(output_path):
    FastLanguageModel.for_inference(model)
    with Path(output_path).open("w", encoding="utf-8") as output:
        for index, task in enumerate(eval_tasks, 1):
            db_id = str(task["db_id"])
            schema = eval_bird.schema_ddl(eval_databases / db_id / f"{db_id}.sqlite")
            messages = [{"role": "user", "content": sql_prompt(schema, task["question"], task.get("evidence", ""))}]
            # enable_thinking=False: empty think block, budget goes to SQL.
            prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)
            # text= keyword: the Qwen3.5 processor's first positional arg is images.
            inputs = tokenizer(text=prompt, return_tensors="pt").to(model.device)
            generated = model.generate(**inputs, max_new_tokens=512, do_sample=False)
            completion = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            sql = eval_bird.strip_fences(completion)
            output.write(json.dumps({"question_id": task["question_id"], "sql": sql}) + "\n")
            output.flush()
            if index % 50 == 0 or index == len(eval_tasks):
                print(f"generated {index}/{len(eval_tasks)}")

def score_predictions(predictions, report):
    subprocess.run([
        "python", str(EVAL_SCRIPT), "--predictions", str(predictions),
        "--bird-dir", str(EVAL_BIRD_DIR), "--report", str(report),
    ], check=True)
    result = json.loads(Path(report).read_text(encoding="utf-8"))
    print(json.dumps(result, indent=2))
    return result

BASELINE_PREDICTIONS = Path("/content/baseline_predictions.jsonl")
generate_predictions(BASELINE_PREDICTIONS)
baseline_report = score_predictions(BASELINE_PREDICTIONS, "/content/baseline_report.json")

## LoRA SFT

Training uses the same prompt shape as evaluation. LoRA rank, alpha, and target modules match the CUAD starting point. Epochs, effective batch size, and learning rate also match it.

In [ ]:
# 8. Format chat examples and train.
import time
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model.print_trainable_parameters()

def format_training_row(row):
    messages = [
        {"role": "user", "content": sql_prompt(row["schema"], row["question"], row.get("evidence", ""))},
        {"role": "assistant", "content": row["sql"]},
    ]
    # enable_thinking=False keeps training text consistent with the eval prompts.
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)}

with TRAIN_JSONL.open(encoding="utf-8") as handle:
    train_rows = [json.loads(line) for line in handle]
train_dataset = Dataset.from_list([format_training_row(row) for row in train_rows])
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="/content/outputs",
        report_to="none",
        bf16=use_bf16,
        fp16=not use_bf16,
    ),
)
started = time.perf_counter()
train_result = trainer.train()
wall_seconds = time.perf_counter() - started
loss_history = [entry["loss"] for entry in trainer.state.log_history if "loss" in entry]
print(f"final logged training loss: {loss_history[-1]:.6f}")
print(f"training wall time: {wall_seconds:.1f} seconds ({wall_seconds / 60:.1f} minutes)")
print(f"trainer train_loss: {train_result.metrics['train_loss']:.6f}")

## Final BIRD evaluation

Run identical generation and scoring after SFT. `QUICK_N` selection, if enabled, is shared with baseline.

In [ ]:
# 9. Score fine-tuned model and print gate comparison.
FINETUNED_PREDICTIONS = Path("/content/finetuned_predictions.jsonl")
generate_predictions(FINETUNED_PREDICTIONS)
finetuned_report = score_predictions(FINETUNED_PREDICTIONS, "/content/finetuned_report.json")
baseline_ex = 100 * baseline_report["overall_ex"]
finetuned_ex = 100 * finetuned_report["overall_ex"]
print("=" * 68)
print(f"Raw Qwen3.5-4B baseline EX : {baseline_ex:.2f}")
print(f"Fine-tuned Qwen3.5-4B EX   : {finetuned_ex:.2f}")
print("Apprentice continue gate   : 66.00")
print("Arctic-R1-7B EX            : 68.50 (published by Snowflake, not our run)")
print(f"Gate result                : {'PASS' if finetuned_ex >= 66.0 else 'STOP'}")
if QUICK_N is not None:
    print("WARNING: QUICK_N ACTIVE — THESE SCORES ARE SMOKE-ONLY AND NOT PUBLISHABLE.")

## Optional persistence

Both blocks stay commented. Save adapter to Drive if needed. Publishing waits for a full-run gate pass and owner decision.

In [ ]:
# 10. Optional: uncomment only after reviewing full-run scores.
# from google.colab import drive
# drive.mount('/content/drive')
# adapter_dir = '/content/drive/MyDrive/apprentice-qwen35-4b-sql-lora'
# model.save_pretrained(adapter_dir)
# tokenizer.save_pretrained(adapter_dir)
# print(f'saved adapter: {adapter_dir}')

# Publishing waits for gate pass + owner call.
# HF_REPO_ID = 'OWNER/apprentice-qwen35-4b-sql-lora'
# model.push_to_hub(HF_REPO_ID, private=True)
# tokenizer.push_to_hub(HF_REPO_ID, private=True)